In [ ]:
# provided a working pip venv or conda environment
# with a recent jupyter and jupyterlab installed,
# h5web will work out of the box to inspect HDF5 files
import glob
import os

import numpy as np
from jupyterlab_h5web import H5Web
from pynxtools_em.examples.oasisb_utils import get_project_id

project_range: tuple[int, int] = (1, 880)


def get_file_size(file_name):
    print(f"{np.around(os.path.getsize(file_name) / 1024 / 1024, decimals=3)} MiB")


print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)
with open("target_directory.txt") as fp:
    trg_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(trg_directory)

In [ ]:
H5Web(
    f"{trg_directory}{os.sep}..{os.sep}pynxtools{os.sep}001.9f8da4417ab3bc4b12c226c8030f7821f6b4aaaa519be4f873911df604ac745c.e5830d322a95d9002380d2a1c8257b7218e357f23d04c717c15a74dd83f64f11.crc.mtex.h5.nxs"
)

## Export default ROI, default IPF plots, and get all phase_name instance values used

pynxtools-microstructure uses a mapping table to identify likely atom_types from NXipf instances.

In [ ]:
# export default plot to image for quickly skimming through and consistency checks
import h5py
from PIL import Image

write_roi: bool = False
write_ipf: bool = False

phase_names: set[str] = set()
for project_name in range(project_range[0], project_range[1] + 1):
    project_id = get_project_id(f"{project_name}")
    pattern = os.path.join(trg_directory, f"{project_id}.*.mtex.h5")
    matching_files = glob.glob(pattern)
    print(f"{project_id}, {len(phase_names)}")
    for matching_file in matching_files:
        # print(matching_file)
        if write_roi:
            prefix = f"{trg_directory.rsplit(os.sep, 1)[0]}{os.sep}gallery{os.sep}{project_id}"
            suffix = f"{matching_file.rsplit(os.sep, 1)[1]}"
            roi_file = f"{prefix}.roi.{suffix}.png"
            if not os.path.isfile(roi_file):
                print(roi_file)
                with h5py.File(matching_file) as h5r:
                    if "/entry1/roi1/ebsd/indexing/roi/data" in h5r:
                        roi = Image.fromarray(
                            h5r["/entry1/roi1/ebsd/indexing/roi/data"][...], mode="L"
                        )
                        roi.save(roi_file)

        if write_ipf:
            with h5py.File(matching_file) as h5r:
                if all(
                    concept in h5r
                    for concept in [
                        "/entry1/roi1/ebsd/indexing/phase1/name",
                        "/entry1/roi1/ebsd/indexing/phase1/ipf1/map/data",
                    ]
                ):
                    ipf = Image.fromarray(
                        h5r["/entry1/roi1/ebsd/indexing/phase1/ipf1/map/data"][...],
                        mode="RGB",
                    )
                    phase_name = (
                        h5r["/entry1/roi1/ebsd/indexing/phase1/name"][()]
                        .decode("utf-8")
                        .replace(" ", "")
                    )
                    ipf.save(f"{prefix}.ipf.{suffix}.{phase_name}.png")

        with h5py.File(matching_file) as h5r:
            parent_group_name = "/entry1/roi1/ebsd/indexing"
            if parent_group_name in h5r:
                for grp_name in h5r[parent_group_name]:
                    if (
                        grp_name.startswith("phase")
                        and f"{parent_group_name}/{grp_name}/name" in h5r
                    ):
                        phase_names.add(
                            h5r[f"{parent_group_name}/{grp_name}/name"][()]
                            .decode("utf-8")
                            .strip()
                        )

        # find all cases where there is only phase0
        # if "/entry1/roi1/ebsd/indexing/phase1" in h5r:
        #     print(matching_file)
print("Batch queue completed")

In [ ]:
from pynxtools_em.examples.ebsd_database import (
    CONCEPT_TO_ATOM_TYPES,
    FREE_TEXT_TO_CONCEPT,
)

for phase_name in sorted(phase_names):
    if phase_name not in FREE_TEXT_TO_CONCEPT:
        print(f"{phase_name} >>>>>>>>>")
    else:
        concept = FREE_TEXT_TO_CONCEPT[phase_name]
        if concept not in CONCEPT_TO_ATOM_TYPES:
            print(f"{concept} -------->")

In [ ]:
# hunt for files with specific type already generated
"""
matching_files = {}
for typ in ("cpr", "crc", "osc", "ang", "ctf"):
    pattern = os.path.join(src_directory, f"*.{typ}")
    matching_files[typ] = glob.glob(pattern)
    print(f"{typ}: {len(matching_files[typ])}")
"""

# for each project present pick one representative (just the first one)
for project_name in range(project_range[0], project_range[1]):
    project_id = get_project_id(f"{project_name}")

    # hunt for slice sets in time and space 3D-EBSD
    """
    pattern = os.path.join(src_directory, f"{project_id}.decompressed.log")
    matching_files = glob.glob(pattern)
    """
    # inspect individual logs
    """
    if len(matching_files) == 1:
        with open(matching_files[0], "r", encoding="utf-8", errors="ignore") as fp:
            lines = fp.readlines()
            if len(lines) > 3:
                print(project_id)
                for line in lines:
                    if line.startswith("INFO") and line.count(">") == 1:
                        print(line.split(">")[0])
    """
    #    matching_files.extend(glob.glob(pattern))
    # pattern = os.path.join(src_directory, f"{project_id}.*
    # hunt for files for following up the analysis process

    pattern = os.path.join(trg_directory, f"{project_id}.*.mtex.h5")
    matching_files = glob.glob(pattern)
    if len(matching_files) > 0:
        # print(f"{project_id}")
        print(
            f'    # "{matching_files[0].replace(f"""{trg_directory}{os.sep}""", "")}",'
        )

In [ ]:
# inspect the log files for anomalities
for project_name in project_range:
    project_id = get_project_id(f"{project_name}")
    pattern = os.path.join(trg_directory, f"{project_id}.log")
    matching_files = glob.glob(pattern)
    if len(matching_files) > 0:
        with open(matching_files[0], encoding="utf-8", errors="ignore") as fp:
            content = fp.readlines()
            for idx, line in enumerate(content):
                if "Error" in line:
                    print(f"{project_id}")
                    print(f"{content[idx : idx + 10]}")

In [ ]:
fname = [
    # "186.f367a28f8b3ad6df24067e22c884dd31f7ff62cb895379a1f18ac7fb031354a3.ctf.mat",
    # "186.f367a28f8b3ad6df24067e22c884dd31f7ff62cb895379a1f18ac7fb031354a3.ctf.mtex",  # Forsterite example
    # "663.b830a98e91683d95981bf367e2d782c04b0ccd82ad3a76c164d09078c8f3580f.ctf.mtex.h5",
]
get_file_size(f"{trg_directory}{os.sep}{fname[0]}")
H5Web(f"{trg_directory}{os.sep}{fname[0]}")

In [ ]:
print(f"{trg_directory.rsplit(os.sep, 1)[0]}{os.sep}gallery")